# 08 · Gold: variables de cada vuelo a T−2 h

**Propósito:** armar la tabla con la que se entrena y evalúa el modelo. Cada fila es una salida programada y cada
columna es una variable que **ya se conocía dos horas antes de la salida** (T−2 h, con T = salida programada en UTC).
Ninguna usa información del propio vuelo que se sepa después.

**Entrada:** `silver/vuelos` y `silver/clima_horario`.

**Salida:** `gold/features_salida/pais=XX/year=AAAA/month=M/` y un resumen en `docs/evidencias/gold/`.

**Requisito:** notebooks 05, 06 y 07 (con las reglas críticas cumplidas).

| Grupo | Variables | Por qué se conoce a T−2 h |
|---|---|---|
| Clima del origen y del destino | viento, visibilidad, techo, temperatura, presión, fenómenos, categoría de vuelo | METAR emitido antes de la hora `k_ref` |
| Red del aeropuerto | salidas ya despegadas en las 2 horas previas, su demora media y la fracción atrasada | salidas reales anteriores a `k_ref` |
| Congestión programada | salidas programadas en el origen en esa hora y en el día | itinerario publicado |
| Calendario | mes, día de la semana, hora local, fin de semana | itinerario |
| Historia del vuelo | demora del mismo número de vuelo el día anterior | solo si ese vuelo salió antes de T−2 h |
| Historia del aeropuerto | tasa de demora por origen y hora, calculada solo con train | datos pasados |

In [ ]:
import os
from pathlib import Path

import yaml

# Carpeta del repositorio (este notebook está en notebooks/)
RAIZ = Path.cwd().parent
with open(RAIZ / "config" / "pipeline.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
tablas = cfg["lake"]["tablas"]

# En la EC2 la variable LAKE_BUCKET trae el nombre del bucket; en una laptop se trabaja sobre data/lake
BUCKET = os.environ.get("LAKE_BUCKET", "").strip()
RAIZ_LOCAL = RAIZ / cfg["lake"]["raiz_local"]
BASE = f"s3a://{BUCKET}" if BUCKET else RAIZ_LOCAL.resolve().as_uri()


def ruta(tabla: str) -> str:
    """Ruta completa de una tabla declarada en config/pipeline.yaml."""
    return f"{BASE}/{tablas[tabla]}"


def meses(desde: str, hasta: str) -> list[tuple[int, int]]:
    """Pares (año, mes) entre dos fechas AAAA-MM, ambas incluidas."""
    anio, mes = map(int, desde.split("-"))
    anio_fin, mes_fin = map(int, hasta.split("-"))
    salida = []
    while (anio, mes) <= (anio_fin, mes_fin):
        salida.append((anio, mes))
        anio, mes = (anio + 1, 1) if mes == 12 else (anio, mes + 1)
    return salida


DESDE, HASTA = cfg["periodo"]["inicio"], cfg["periodo"]["fin"]
print("Destino:", f"s3://{BUCKET}" if BUCKET else "data/lake (local)")
print("Periodo:", DESDE, "a", HASTA)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import pandas as pd

# En la EC2 Spark toma la configuración del clúster (spark-defaults.conf); en una laptop corre en modo local
constructor = SparkSession.builder.appName("latam-bigdata-gold-features")
if not BUCKET:
    constructor = (
        constructor.master("local[*]")
        .config("spark.sql.shuffle.partitions", str(cfg["spark"]["particiones_shuffle_local"]))
        .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "3g"))
    )
spark = (
    constructor.config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")


def en_periodo(df, col_anio="year", col_mes="month"):
    """Deja solo las filas cuyo año y mes están dentro del periodo de config/pipeline.yaml."""
    ym = F.format_string("%04d-%02d", F.col(col_anio), F.col(col_mes))
    return df.where((ym >= DESDE) & (ym <= HASTA))

In [ ]:
EVID = RAIZ / "docs" / "evidencias" / "gold"
EVID.mkdir(parents=True, exist_ok=True)
TRAIN_FIN = cfg["particion_temporal"]["train_fin"]
VALID_FIN = cfg["particion_temporal"]["valid_fin"]
HORIZONTE = cfg["horizonte_horas"]
UMBRAL_AMBAR = cfg["umbrales_demora_min"]["ambar"]
MIN_VUELOS_TASA = cfg["features"]["min_vuelos_tasa_hist"]
NUMERICAS = cfg["features"]["numericas"]
CATEGORICAS = cfg["features"]["categoricas"]

vuelos = en_periodo(spark.read.parquet(ruta("silver_vuelos")))
clima_horario = spark.read.parquet(ruta("silver_clima_horario"))

## 1. Vuelos que entran al modelo y hora de referencia

Se excluyen cancelados, desviados y demoras atípicas: el modelo predice la demora de salida de un vuelo que opera.
`k_ref` es la hora en punto anterior o igual a T−2 h. Toda la información que se une al vuelo es anterior a `k_ref`.

In [ ]:
ym = F.format_string("%04d-%02d", F.col("year"), F.col("month"))
base = (
    vuelos.where(~F.col("cancelado") & ~F.col("desviado") & ~F.col("es_atipico") & F.col("y_demora15").isNotNull())
    .withColumn("t_menos_2h", F.col("salida_prog_utc") - F.expr(f"INTERVAL {HORIZONTE} HOURS"))
    .withColumn("k_ref", F.date_trunc("hour", "t_menos_2h"))
    .withColumn("particion", F.when(ym <= TRAIN_FIN, "train").when(ym <= VALID_FIN, "valid").otherwise("test"))
)

## 2. Clima del origen y del destino

`clima_horario` en la hora `k_ref` ya contiene la última observación **estrictamente anterior** a `k_ref`. Este join
es grande contra grande (millones de vuelos contra millones de horas), así que Spark lo resuelve ordenando ambos lados
por la clave (*sort-merge join*). El *broadcast* se reserva para tablas chicas, como la de zonas horarias del
notebook 05 o la tasa histórica de más abajo, que caben enteras en la memoria de cada nodo.

In [ ]:
CLIMA_ORIGEN = [
    "obs_utc", "viento_kt", "rafaga_kt", "visibilidad_m", "techo_ft", "temp_c", "spread_t_td", "presion_hpa",
    "presion_tendencia_3h", "fen_niebla", "fen_neblina", "fen_tormenta", "fen_lluvia", "fen_nieve", "tiene_tempo",
    "horas_desde_obs", "categoria_vuelo",
]
CLIMA_DESTINO = ["viento_kt", "visibilidad_m", "techo_ft", "fen_niebla", "fen_tormenta", "fen_lluvia"]

clima_o = clima_horario.select(
    F.col("aeropuerto").alias("origen"), F.col("k").alias("k_ref"), *[F.col(c).alias(f"o_{c}") for c in CLIMA_ORIGEN]
)
clima_d = clima_horario.select(
    F.col("aeropuerto").alias("destino"), F.col("k").alias("k_ref"), *[F.col(c).alias(f"d_{c}") for c in CLIMA_DESTINO]
)
features = (
    base.join(clima_o, ["origen", "k_ref"], "left")
    .join(clima_d, ["destino", "k_ref"], "left")
    .withColumn("clima_disponible", F.col("o_obs_utc").isNotNull().cast("int"))
)

## 3. Estado de la red en el aeropuerto de origen

Para cada vuelo se mira qué pasó con las salidas del mismo aeropuerto programadas en las dos horas anteriores a
`k_ref` (`[k_ref − 2 h, k_ref)`) que **ya despegaron antes de `k_ref`**: cuántas son, su demora media y qué fracción
salió con 15 minutos o más. Es la propagación de demoras que se puede observar sin conocer la rotación de cada avión.

Se calcula sin cruzar vuelos contra vuelos: cada salida se agrega una vez para la hora `k_ref` siguiente y otra para
la subsiguiente, siempre que ya hubiera despegado antes de esa `k_ref`.

In [ ]:
despegados = (
    vuelos.where(F.col("salida_real_utc").isNotNull() & ~F.col("es_atipico"))
    .select("pais", "origen", F.date_trunc("hour", "salida_prog_utc").alias("hora_prog"), "salida_real_utc",
            "demora_salida_min")
)
aportes = None
for horas in (1, 2):
    parte = (
        despegados.withColumn("k_ref", F.col("hora_prog") + F.expr(f"INTERVAL {horas} HOURS"))
        .where(F.col("salida_real_utc") < F.col("k_ref"))
    )
    aportes = parte if aportes is None else aportes.unionByName(parte)
red = aportes.groupBy("pais", "origen", "k_ref").agg(
    F.count("*").alias("red_n_salidas"),
    F.round(F.avg("demora_salida_min"), 2).alias("red_demora_media"),
    F.round(F.avg((F.col("demora_salida_min") >= UMBRAL_AMBAR).cast("int")), 4).alias("red_frac_atrasadas"),
)
features = features.join(red, ["pais", "origen", "k_ref"], "left").fillna({"red_n_salidas": 0})

## 4. Congestión programada y calendario

La congestión cuenta **todas** las salidas programadas del origen (también las que después se cancelaron), porque
el itinerario se publica con anticipación.

In [ ]:
congestion_hora = vuelos.groupBy("pais", "origen", "fecha_local", "hora_local_salida").agg(
    F.count("*").alias("congestion_hora"))
congestion_dia = vuelos.groupBy("pais", "origen", "fecha_local").agg(F.count("*").alias("congestion_dia"))
features = (
    features.join(congestion_hora, ["pais", "origen", "fecha_local", "hora_local_salida"], "left")
    .join(congestion_dia, ["pais", "origen", "fecha_local"], "left")
    .withColumn("mes", F.month("fecha_local"))
    .withColumn("dia_semana", F.dayofweek("fecha_local"))
    .withColumn("hora_local", F.col("hora_local_salida"))
    .withColumn("fin_de_semana", F.col("dia_semana").isin(1, 7).cast("int"))
)

## 5. El mismo vuelo el día anterior

Se toma la demora de salida del mismo número de vuelo, aerolínea y origen del día anterior. Solo se usa si ese
vuelo ya había despegado antes de T−2 h; si no, queda vacía y `prev_dia_conocido` = 0.

In [ ]:
CLAVE_VUELO = ["pais", "aerolinea", "numero_vuelo", "origen"]
ayer = (
    vuelos.where(F.col("demora_salida_min").isNotNull() & ~F.col("es_atipico"))
    .withColumn("fecha_local", F.date_add("fecha_local", 1))
    .groupBy(*CLAVE_VUELO, "fecha_local")
    .agg(F.avg("demora_salida_min").alias("_demora_ayer"), F.max("salida_real_utc").alias("_real_ayer"))
)
conocido = F.col("_real_ayer") <= F.col("t_menos_2h")
features = (
    features.join(ayer, [*CLAVE_VUELO, "fecha_local"], "left")
    .withColumn("prev_dia_demora", F.when(conocido, F.round("_demora_ayer", 1)))
    .withColumn("prev_dia_conocido", F.coalesce(conocido, F.lit(False)).cast("int"))
    .drop("_demora_ayer", "_real_ayer")
)

## 6. Tasa histórica de demora por aeropuerto y hora

Es la proporción de salidas con 15 minutos o más para cada (país, origen, hora local), calculada **solo con
train**. Para las filas de train se excluye su propio mes: así la tasa de un vuelo nunca incluye su propia etiqueta.
Si hay menos de `min_vuelos_tasa_hist` vuelos de referencia, queda vacía.

In [ ]:
CLAVE_TASA = ["pais", "origen", "hora_local_salida"]
por_mes = (
    base.where(F.col("particion") == "train")
    .groupBy(*CLAVE_TASA, "year", "month").agg(F.sum("y_demora15").alias("_s_mes"), F.count("*").alias("_n_mes"))
)
total = por_mes.groupBy(*CLAVE_TASA).agg(F.sum("_s_mes").alias("_s"), F.sum("_n_mes").alias("_n"))
es_train = F.col("particion") == "train"
s = F.when(es_train, F.col("_s") - F.coalesce("_s_mes", F.lit(0))).otherwise(F.col("_s"))
n = F.when(es_train, F.col("_n") - F.coalesce("_n_mes", F.lit(0))).otherwise(F.col("_n"))
features = (
    features.join(F.broadcast(total), CLAVE_TASA, "left")
    .join(F.broadcast(por_mes), [*CLAVE_TASA, "year", "month"], "left")
    .withColumn("tasa_hist_origen_hora", F.when(n >= MIN_VUELOS_TASA, F.round(s / n, 4)))
    .drop("_s", "_n", "_s_mes", "_n_mes")
)

## 7. Verificación contra fugas de información

Antes de escribir se comprueba que:

1. ninguna columna prohibida (lo que se sabe después de la salida) está en la lista de variables del modelo;
2. ninguna fila usa una hora de referencia posterior a T−2 h;
3. ninguna fila usa un METAR igual o posterior a `k_ref`.

In [ ]:
PROHIBIDAS = {
    "salida_real_utc", "llegada_real_utc", "demora_salida_min", "demora_llegada_min", "cancelado", "desviado",
    "es_atipico", "y_demora15", "y_demora60",
}
variables = set(NUMERICAS) | set(CATEGORICAS)
assert not variables & PROHIBIDAS, f"Variables prohibidas en la lista del modelo: {variables & PROHIBIDAS}"
assert not [v for v in variables if v.startswith(("demora_", "salida_real", "llegada_real"))]
faltan = variables - set(features.columns)
assert not faltan, f"Variables de config que no existen en la tabla: {faltan}"

COLUMNAS = [
    "id_vuelo", "pais", "aerolinea", "es_latam", "numero_vuelo", "origen", "destino", "salida_prog_utc",
    "fecha_local", "k_ref", "o_obs_utc", "particion", "y_demora15", "y_demora60", "demora_salida_min",
    *sorted(variables - {"pais", "aerolinea"}), "year", "month",
]
gold = features.select(*COLUMNAS)
violaciones = gold.where(
    (F.col("k_ref") > F.col("salida_prog_utc") - F.expr(f"INTERVAL {HORIZONTE} HOURS"))
    | (F.col("o_obs_utc") >= F.col("k_ref"))
).count()
assert violaciones == 0, f"{violaciones} filas usan información posterior a T-2 h"
print("Sin fugas: variables permitidas, k_ref <= T-2 h y METAR anterior a k_ref en todas las filas.")

`demora_salida_min` se guarda en gold solo como dato de referencia (para análisis y para un posible modelo de
minutos en la segunda unidad); no está en la lista de variables del modelo.

## 8. Escritura

In [ ]:
(
    gold.repartition("pais", "year", "month")
    .write.mode("overwrite").partitionBy("pais", "year", "month").parquet(ruta("gold_features"))
)
gold = spark.read.parquet(ruta("gold_features"))
resumen = (
    gold.groupBy("pais", "particion").agg(
        F.count("*").alias("vuelos"),
        F.round(100 * F.avg("y_demora15"), 2).alias("pct_demora15"),
        F.round(100 * F.avg("clima_disponible"), 2).alias("pct_con_clima"),
        F.round(100 * F.avg((F.col("red_n_salidas") > 0).cast("int")), 2).alias("pct_con_red"),
        F.round(100 * F.avg("prev_dia_conocido"), 2).alias("pct_con_dia_anterior"),
        F.round(100 * F.count("tasa_hist_origen_hora") / F.count("*"), 2).alias("pct_con_tasa_hist"),
    ).orderBy("pais", "particion").toPandas()
)
resumen.to_csv(EVID / "resumen_features.csv", index=False)
resumen

In [ ]:
gold.where(F.col("es_latam")).select("id_vuelo", "origen", "salida_prog_utc", "k_ref", "o_obs_utc",
                                     "o_categoria_vuelo", "red_n_salidas", "red_frac_atrasadas",
                                     "prev_dia_demora", "tasa_hist_origen_hora", "y_demora15").limit(8).toPandas()

In [ ]:
spark.stop()